# Split Authors by Name Normalization

One-time data repair: splits author profiles where the improved `CreateParsedNamesNormalized` reveals that work names no longer match the author's canonical `full_name`.

**Prerequisites:**
1. `CreateParsedNamesNormalized` has been run with the improved parser
2. `CreateAuthors` has been run to rebuild `authors_for_matching` with new normalization
3. `AnalyzeAuthorNameMismatches` has been run — `name_mismatch_block_key` and `name_mismatch_pattern` tables exist

**Downstream propagation:** After this notebook runs, the next end-to-end pipeline run automatically rebuilds `work_authorships`, `openalex_works`, and ES sync via `updated_at` on `work_authors`.

### Step 1: Identify split candidates

Pull the mismatched assignments from the analysis table.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.split_candidates AS
-- Pass 1: Block-key mismatches (different last name or first initial)
SELECT
    work_id, author_sequence, author_id, raw_author_name, author_full_name,
    NULL AS w_first, w_first_initial, NULL AS w_middle, NULL AS w_middle_initials, w_last
FROM openalex.authors.name_mismatch_block_key
WHERE work_block_key IS NOT NULL AND author_block_key IS NOT NULL
  AND work_block_key != author_block_key
UNION ALL
-- Pass 2: Pattern-level mismatches (same block_key, no pattern matches)
SELECT
    work_id, author_sequence, author_id, raw_author_name, author_full_name,
    w_first, w_first_initial, w_middle, w_middle_initials, w_last
FROM openalex.authors.name_mismatch_pattern

### Step 1b: Diagnostic — split candidate counts

In [0]:
SELECT
    COUNT(*) AS total_split_candidates,
    COUNT(DISTINCT author_id) AS affected_authors
FROM openalex.authors.split_candidates

### Step 2: Determine majority cluster per author

For each affected author, group ALL their works by normalized name cluster. The largest cluster keeps the existing `author_id`. Ties broken by matching the author's current `full_name`.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.split_author_clusters AS
WITH all_works_on_affected_authors AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id,
        wa.raw_author_name,
        pn.normalized_first,
        pn.normalized_first_initial,
        pn.normalized_last,
        CASE
            WHEN pn.normalized_first IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first, ' ', pn.normalized_last)
            WHEN pn.normalized_first_initial IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first_initial, ' ', pn.normalized_last)
            ELSE LOWER(TRIM(wa.raw_author_name))
        END AS name_cluster_key
    FROM openalex.works.work_authors wa
    LEFT JOIN openalex.authors.parsed_names_normalized pn
        ON TRIM(wa.raw_author_name) = pn.raw_author_name
    WHERE wa.author_id IN (SELECT DISTINCT author_id FROM openalex.authors.split_candidates)
),
cluster_sizes AS (
    SELECT
        wk.author_id,
        wk.name_cluster_key,
        COUNT(*) AS cluster_work_count,
        MAX(CASE
            WHEN CASE
                    WHEN apn.normalized_first IS NOT NULL AND apn.normalized_last IS NOT NULL
                    THEN CONCAT(apn.normalized_first, ' ', apn.normalized_last)
                    WHEN apn.normalized_first_initial IS NOT NULL AND apn.normalized_last IS NOT NULL
                    THEN CONCAT(apn.normalized_first_initial, ' ', apn.normalized_last)
                    ELSE LOWER(TRIM(a.full_name))
                 END = wk.name_cluster_key
            THEN 1 ELSE 0
        END) AS matches_full_name
    FROM all_works_on_affected_authors wk
    JOIN openalex.authors.authors a ON wk.author_id = a.id
    LEFT JOIN openalex.authors.parsed_names_normalized apn
        ON TRIM(a.full_name) = apn.raw_author_name
    GROUP BY wk.author_id, wk.name_cluster_key
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY author_id
            ORDER BY cluster_work_count DESC, matches_full_name DESC, name_cluster_key
        ) AS rank
    FROM cluster_sizes
)
SELECT
    author_id,
    name_cluster_key,
    cluster_work_count,
    rank,
    CASE WHEN rank = 1 THEN TRUE ELSE FALSE END AS keeps_author_id
FROM ranked

### Step 2b: Diagnostic — cluster distribution

In [0]:
SELECT
    COUNT(DISTINCT author_id) AS total_authors,
    SUM(CASE WHEN keeps_author_id THEN cluster_work_count ELSE 0 END) AS works_staying,
    SUM(CASE WHEN NOT keeps_author_id THEN cluster_work_count ELSE 0 END) AS works_splitting,
    COUNT_IF(NOT keeps_author_id) AS minority_clusters
FROM openalex.authors.split_author_clusters

### Step 3: Build split work removals

Works NOT in the majority cluster get split off. Enrich with parsed name components, block_key, institutions, topics, and sources for re-matching.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.split_work_removals AS
WITH minority_clusters AS (
    SELECT author_id, name_cluster_key
    FROM openalex.authors.split_author_clusters
    WHERE NOT keeps_author_id
),
base_removals AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id AS old_author_id,
        wa.raw_author_name,
        pn.normalized_first AS pn_first,
        pn.normalized_first_initial AS pn_first_initial,
        pn.normalized_middle AS pn_middle,
        pn.normalized_middle_initials AS pn_middle_initials,
        pn.normalized_last AS pn_last,
        CASE
            WHEN pn.normalized_last IS NULL THEN NULL
            WHEN pn.normalized_first_initial IS NULL THEN pn.normalized_last
            ELSE CONCAT(pn.normalized_first_initial, ' ', pn.normalized_last)
        END AS block_key,
        CASE
            WHEN pn.normalized_first IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first, ' ', pn.normalized_last)
            WHEN pn.normalized_first_initial IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first_initial, ' ', pn.normalized_last)
            ELSE LOWER(TRIM(wa.raw_author_name))
        END AS name_cluster_key
    FROM openalex.works.work_authors wa
    LEFT JOIN openalex.authors.parsed_names_normalized pn
        ON TRIM(wa.raw_author_name) = pn.raw_author_name
    JOIN minority_clusters mc
        ON wa.author_id = mc.author_id
    WHERE CASE
            WHEN pn.normalized_first IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first, ' ', pn.normalized_last)
            WHEN pn.normalized_first_initial IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first_initial, ' ', pn.normalized_last)
            ELSE LOWER(TRIM(wa.raw_author_name))
          END = mc.name_cluster_key
),
-- Resolve institution IDs from work_authors.raw_affiliation_strings
exploded_ras AS (
    SELECT br.work_id, br.author_sequence, t.raw_affiliation_string
    FROM base_removals br
    JOIN openalex.works.work_authors wa
        ON br.work_id = wa.work_id AND br.author_sequence = wa.author_sequence
    LATERAL VIEW OUTER EXPLODE(wa.raw_affiliation_strings) t AS raw_affiliation_string
),
resolved_direct_ids AS (
    SELECT
        e.work_id, e.author_sequence,
        CASE
            WHEN e.raw_affiliation_string IS NULL THEN NULL
            WHEN asl.institution_ids IS NOT NULL AND SIZE(asl.institution_ids) > 0
                AND NOT (SIZE(asl.institution_ids) = 1 AND asl.institution_ids[0] = -1)
                THEN asl.institution_ids
            ELSE NULL
        END AS direct_ids
    FROM exploded_ras e
    LEFT JOIN openalex.institutions.raw_affiliation_strings_institutions_mv asl
        ON e.raw_affiliation_string = asl.raw_affiliation_string
),
expanded_for_matching AS (
    SELECT
        r.work_id,
        r.author_sequence,
        ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(
            FLATTEN(ARRAY(
                FILTER(ARRAY(r.inst_id_scalar), x -> x IS NOT NULL),
                COALESCE(anc.ancestors, ARRAY())
            ))
        ))) AS matching_institution_ids
    FROM (
        SELECT work_id, author_sequence, EXPLODE_OUTER(direct_ids) AS inst_id_scalar
        FROM resolved_direct_ids
    ) r
    LEFT JOIN (
        SELECT institution_id, lineage_ids AS ancestors
        FROM openalex.institutions.institution_ancestors
    ) anc ON CAST(r.inst_id_scalar AS BIGINT) = anc.institution_id
    GROUP BY r.work_id, r.author_sequence
)
SELECT
    br.*,
    TRANSFORM(COALESCE(efm.matching_institution_ids, ARRAY()), x -> CONCAT('https://openalex.org/I', CAST(x AS STRING))) AS institution_ids,
    COALESCE(efm.matching_institution_ids, ARRAY()) AS all_institution_ids,
    COALESCE(TRANSFORM(wtf.topics, t -> t.id), ARRAY()) AS topic_ids,
    ARRAY_DISTINCT(
        TRANSFORM(FILTER(COALESCE(w.locations, ARRAY()), x -> x.source.id IS NOT NULL), x -> x.source.id)
    ) AS work_source_ids
FROM base_removals br
LEFT JOIN expanded_for_matching efm
    ON br.work_id = efm.work_id AND br.author_sequence = efm.author_sequence
LEFT JOIN openalex.works.work_topics wtf ON br.work_id = wtf.work_id
LEFT JOIN openalex.works.openalex_works w ON br.work_id = w.id

### Step 3b: Diagnostic — removal counts

In [0]:
SELECT
    COUNT(*) AS total_works_to_split,
    COUNT(DISTINCT old_author_id) AS distinct_old_authors,
    COUNT(DISTINCT name_cluster_key) AS distinct_name_clusters
FROM openalex.authors.split_work_removals

### Step 4: Re-match split-off works against existing authors

Uses the full 8-pattern + signal matching from MatchAuthors. Excludes the `old_author_id` from candidates to avoid circular matching.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.split_rematched AS
WITH
blocked_candidates AS (
    SELECT
        r.work_id,
        r.author_sequence,
        r.raw_author_name,
        r.old_author_id,
        r.pn_first,
        r.pn_first_initial,
        r.pn_middle,
        r.pn_middle_initials,
        r.pn_last,
        r.block_key,
        r.institution_ids,
        r.topic_ids,
        r.work_source_ids,
        r.all_institution_ids,
        r.name_cluster_key,
        alm.author_id,
        alm.display_name AS candidate_display_name,
        alm.normalized_first AS cand_first,
        alm.normalized_first_initial AS cand_first_initial,
        alm.normalized_middle AS cand_middle,
        alm.normalized_middle_initials AS cand_middle_initials,
        alm.normalized_last AS cand_last,
        alm.institution_ids AS candidate_institution_ids,
        alm.topic_ids AS candidate_topic_ids,
        alm.source_ids AS candidate_source_ids,
        alm.works_count
    FROM openalex.authors.split_work_removals r
    LEFT JOIN openalex.authors.authors_for_matching alm
        ON alm.block_key = r.block_key
        AND alm.author_id != r.old_author_id
),
with_match_signals AS (
    SELECT
        *,
        NAMED_STRUCT('id', author_id, 'display_name', candidate_display_name) AS candidate_obj,
        (SIZE(institution_ids) > 0 AND SIZE(candidate_institution_ids) > 0
         AND ARRAYS_OVERLAP(candidate_institution_ids, institution_ids)) AS has_inst,
        (SIZE(topic_ids) > 0 AND SIZE(candidate_topic_ids) > 0
         AND ARRAYS_OVERLAP(candidate_topic_ids, topic_ids)) AS has_topic,
        (SIZE(work_source_ids) > 0 AND SIZE(candidate_source_ids) > 0
         AND ARRAYS_OVERLAP(candidate_source_ids, work_source_ids)) AS has_source
    FROM blocked_candidates
),
with_name_matches AS (
    SELECT
        *,
        -- Pattern 1: Exact Full Name
        (pn_first IS NOT NULL AND pn_middle IS NOT NULL AND cand_first IS NOT NULL AND cand_middle IS NOT NULL
         AND pn_first = cand_first AND pn_middle = cand_middle AND pn_last = cand_last
        ) AS pattern_1_exact_full,
        -- Pattern 2: Exact First, Middle Initial match
        (pn_first IS NOT NULL AND pn_middle IS NULL AND pn_middle_initials IS NOT NULL
         AND cand_first IS NOT NULL AND pn_first = cand_first AND pn_last = cand_last
         AND (cand_middle_initials IS NULL OR SUBSTRING(pn_middle_initials, 1, 1) = SUBSTRING(cand_middle_initials, 1, 1))
        ) AS pattern_2_exact_first_mid_init,
        -- Pattern 3: Initials match to Full
        (pn_first IS NULL AND pn_first_initial IS NOT NULL AND pn_middle_initials IS NOT NULL
         AND cand_first IS NOT NULL AND cand_middle_initials IS NOT NULL
         AND pn_first_initial = cand_first_initial
         AND SUBSTRING(pn_middle_initials, 1, 1) = SUBSTRING(cand_middle_initials, 1, 1)
         AND pn_last = cand_last
        ) AS pattern_3_init_mid_init,
        -- Pattern 4: Both have only initials
        (pn_first IS NULL AND pn_first_initial IS NOT NULL AND cand_first IS NULL AND cand_first_initial IS NOT NULL
         AND pn_middle IS NULL AND pn_middle_initials IS NOT NULL AND cand_middle IS NULL AND cand_middle_initials IS NOT NULL
         AND pn_first_initial = cand_first_initial
         AND SUBSTRING(pn_middle_initials, 1, 1) = SUBSTRING(cand_middle_initials, 1, 1)
         AND pn_last = cand_last
        ) AS pattern_4_first_init_mid_init,
        -- Pattern 5: Exact First + Last, no middle
        (pn_first IS NOT NULL AND cand_first IS NOT NULL
         AND pn_first = cand_first AND pn_last = cand_last
         AND pn_middle_initials IS NULL
        ) AS pattern_5_exact_first_last,
        -- Pattern 6: First Initial to Full
        (pn_first IS NULL AND pn_first_initial IS NOT NULL AND pn_middle_initials IS NULL
         AND cand_first IS NOT NULL
         AND pn_first_initial = cand_first_initial AND pn_last = cand_last
        ) AS pattern_6_first_init_to_full,
        -- Pattern 7: Both have only first initial, no middle
        (pn_first IS NULL AND pn_first_initial IS NOT NULL AND cand_first IS NULL AND cand_first_initial IS NOT NULL
         AND pn_middle_initials IS NULL AND cand_middle_initials IS NULL
         AND pn_first_initial = cand_first_initial AND pn_last = cand_last
        ) AS pattern_7_first_init_last,
        -- Pattern 8: Full Name to Initial
        (pn_first IS NOT NULL AND cand_first IS NULL AND cand_first_initial IS NOT NULL
         AND pn_first_initial = cand_first_initial AND pn_last = cand_last
        ) AS pattern_8_full_to_init
    FROM with_match_signals
),
with_any_name_match AS (
    SELECT *,
        (pattern_1_exact_full OR pattern_2_exact_first_mid_init OR pattern_3_init_mid_init OR
         pattern_4_first_init_mid_init OR pattern_5_exact_first_last OR pattern_6_first_init_to_full OR
         pattern_7_first_init_last OR pattern_8_full_to_init) AS any_name_match
    FROM with_name_matches
),
aggregated_counts AS (
    SELECT
        work_id, author_sequence, raw_author_name, block_key, old_author_id,
        institution_ids, topic_ids, work_source_ids, all_institution_ids, name_cluster_key,
        pn_first, pn_first_initial, pn_middle, pn_middle_initials, pn_last,

        -- STRATEGY 1: Name Only
        COUNT_IF(pattern_1_exact_full) AS s1_n1, COUNT_IF(pattern_2_exact_first_mid_init) AS s1_n2,
        COUNT_IF(pattern_3_init_mid_init) AS s1_n3, COUNT_IF(pattern_4_first_init_mid_init) AS s1_n4,
        COUNT_IF(pattern_5_exact_first_last) AS s1_n5, COUNT_IF(pattern_6_first_init_to_full) AS s1_n6,
        COUNT_IF(pattern_7_first_init_last) AS s1_n7, COUNT_IF(pattern_8_full_to_init) AS s1_n8,
        -- STRATEGY 2: Name + Institution
        COUNT_IF(pattern_1_exact_full AND has_inst) AS s2_n1, COUNT_IF(pattern_2_exact_first_mid_init AND has_inst) AS s2_n2,
        COUNT_IF(pattern_3_init_mid_init AND has_inst) AS s2_n3, COUNT_IF(pattern_4_first_init_mid_init AND has_inst) AS s2_n4,
        COUNT_IF(pattern_5_exact_first_last AND has_inst) AS s2_n5, COUNT_IF(pattern_6_first_init_to_full AND has_inst) AS s2_n6,
        COUNT_IF(pattern_7_first_init_last AND has_inst) AS s2_n7, COUNT_IF(pattern_8_full_to_init AND has_inst) AS s2_n8,
        -- STRATEGY 6: Name + Inst + Source
        COUNT_IF(pattern_1_exact_full AND has_inst AND has_source) AS s6_n1,
        COUNT_IF(pattern_2_exact_first_mid_init AND has_inst AND has_source) AS s6_n2,
        COUNT_IF(pattern_5_exact_first_last AND has_inst AND has_source) AS s6_n5,
        COUNT_IF(pattern_6_first_init_to_full AND has_inst AND has_source) AS s6_n6,
        COUNT_IF(pattern_7_first_init_last AND has_inst AND has_source) AS s6_n7,
        COUNT_IF(pattern_8_full_to_init AND has_inst AND has_source) AS s6_n8,
        -- STRATEGY 4: Name + Inst + Topic
        COUNT_IF(pattern_1_exact_full AND has_inst AND has_topic) AS s4_n1,
        COUNT_IF(pattern_2_exact_first_mid_init AND has_inst AND has_topic) AS s4_n2,
        COUNT_IF(pattern_5_exact_first_last AND has_inst AND has_topic) AS s4_n5,
        COUNT_IF(pattern_6_first_init_to_full AND has_inst AND has_topic) AS s4_n6,
        COUNT_IF(pattern_7_first_init_last AND has_inst AND has_topic) AS s4_n7,
        COUNT_IF(pattern_8_full_to_init AND has_inst AND has_topic) AS s4_n8,
        -- STRATEGY 5: Name + Source
        COUNT_IF(pattern_1_exact_full AND has_source) AS s5_n1,
        COUNT_IF(pattern_2_exact_first_mid_init AND has_source) AS s5_n2,
        COUNT_IF(pattern_5_exact_first_last AND has_source) AS s5_n5,
        COUNT_IF(pattern_6_first_init_to_full AND has_source) AS s5_n6,
        COUNT_IF(pattern_7_first_init_last AND has_source) AS s5_n7,
        COUNT_IF(pattern_8_full_to_init AND has_source) AS s5_n8,
        -- STRATEGY 3: Name + Topic
        COUNT_IF(pattern_1_exact_full AND has_topic) AS s3_n1,
        COUNT_IF(pattern_2_exact_first_mid_init AND has_topic) AS s3_n2,
        COUNT_IF(pattern_5_exact_first_last AND has_topic) AS s3_n5,

        -- CAPTURE MATCHED OBJECTS
        MAX(CASE WHEN pattern_1_exact_full THEN candidate_obj END) AS match_s1_n1,
        MAX(CASE WHEN pattern_2_exact_first_mid_init THEN candidate_obj END) AS match_s1_n2,
        MAX(CASE WHEN pattern_5_exact_first_last THEN candidate_obj END) AS match_s1_n5,
        MAX(CASE WHEN pattern_1_exact_full AND has_inst THEN candidate_obj END) AS match_s2_n1,
        MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst THEN candidate_obj END) AS match_s2_n2,
        MAX(CASE WHEN pattern_5_exact_first_last AND has_inst THEN candidate_obj END) AS match_s2_n5,
        MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst THEN candidate_obj END) AS match_s2_n6,
        MAX(CASE WHEN pattern_8_full_to_init AND has_inst THEN candidate_obj END) AS match_s2_n8,
        MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n1,
        MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n2,
        MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n5,
        MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n6,
        MAX(CASE WHEN pattern_8_full_to_init AND has_inst AND has_source THEN candidate_obj END) AS match_s6_n8,
        MAX(CASE WHEN pattern_1_exact_full AND has_source THEN candidate_obj END) AS match_s5_n1,
        MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_source THEN candidate_obj END) AS match_s5_n2,
        MAX(CASE WHEN pattern_5_exact_first_last AND has_source THEN candidate_obj END) AS match_s5_n5,
        MAX(CASE WHEN pattern_6_first_init_to_full AND has_source THEN candidate_obj END) AS match_s5_n6,
        MAX(CASE WHEN pattern_8_full_to_init AND has_source THEN candidate_obj END) AS match_s5_n8,
        MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n1,
        MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n2,
        MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n5,
        MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n6,
        MAX(CASE WHEN pattern_8_full_to_init AND has_inst AND has_topic THEN candidate_obj END) AS match_s4_n8,
        MAX(CASE WHEN pattern_1_exact_full AND has_topic THEN candidate_obj END) AS match_s3_n1,
        MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_topic THEN candidate_obj END) AS match_s3_n2,
        MAX(CASE WHEN pattern_5_exact_first_last AND has_topic THEN candidate_obj END) AS match_s3_n5,

        COUNT(author_id) AS total_candidates_in_block,
        COUNT_IF(any_name_match) AS total_name_matches

    FROM with_any_name_match
    GROUP BY work_id, author_sequence, raw_author_name, block_key, old_author_id,
             institution_ids, topic_ids, work_source_ids, all_institution_ids, name_cluster_key,
             pn_first, pn_first_initial, pn_middle, pn_middle_initials, pn_last
),
final_decision AS (
    SELECT
        work_id, author_sequence, block_key, raw_author_name, old_author_id,
        institution_ids, all_institution_ids, work_source_ids, name_cluster_key,
        pn_first, pn_first_initial, pn_last,

        CASE
            WHEN (
                s1_n1=1 OR s1_n2=1 OR s1_n5=1 OR
                s6_n1=1 OR s6_n2=1 OR s6_n5=1 OR s6_n6=1 OR s6_n8=1 OR
                s2_n1=1 OR s2_n2=1 OR s2_n5=1 OR s2_n6=1 OR s2_n8=1 OR
                s4_n1=1 OR s4_n2=1 OR s4_n5=1 OR s4_n6=1 OR s4_n8=1 OR
                s5_n1=1 OR s5_n2=1 OR s5_n5=1 OR s5_n6=1 OR s5_n8=1 OR
                s3_n1=1 OR s3_n2=1 OR s3_n5=1
            ) THEN 'MATCHED'
            WHEN total_candidates_in_block = 0 THEN 'NO_CANDIDATES'
            ELSE 'AMBIGUOUS'
        END AS match_outcome,

        CASE
            WHEN s1_n1 = 1 THEN match_s1_n1.id WHEN s1_n2 = 1 THEN match_s1_n2.id WHEN s1_n5 = 1 THEN match_s1_n5.id
            WHEN s6_n1 = 1 THEN match_s6_n1.id WHEN s6_n2 = 1 THEN match_s6_n2.id WHEN s6_n5 = 1 THEN match_s6_n5.id
            WHEN s6_n6 = 1 THEN match_s6_n6.id WHEN s6_n8 = 1 THEN match_s6_n8.id
            WHEN s2_n1 = 1 THEN match_s2_n1.id WHEN s2_n2 = 1 THEN match_s2_n2.id WHEN s2_n5 = 1 THEN match_s2_n5.id
            WHEN s2_n6 = 1 THEN match_s2_n6.id WHEN s2_n8 = 1 THEN match_s2_n8.id
            WHEN s4_n1 = 1 THEN match_s4_n1.id WHEN s4_n2 = 1 THEN match_s4_n2.id WHEN s4_n5 = 1 THEN match_s4_n5.id
            WHEN s4_n6 = 1 THEN match_s4_n6.id WHEN s4_n8 = 1 THEN match_s4_n8.id
            WHEN s5_n1 = 1 THEN match_s5_n1.id WHEN s5_n2 = 1 THEN match_s5_n2.id WHEN s5_n5 = 1 THEN match_s5_n5.id
            WHEN s5_n6 = 1 THEN match_s5_n6.id WHEN s5_n8 = 1 THEN match_s5_n8.id
            WHEN s3_n1 = 1 THEN match_s3_n1.id WHEN s3_n2 = 1 THEN match_s3_n2.id WHEN s3_n5 = 1 THEN match_s3_n5.id
            ELSE NULL
        END AS existing_author_id

    FROM aggregated_counts
)
SELECT * FROM final_decision

### Step 4b: Diagnostic — re-match rates

In [0]:
SELECT
    match_outcome,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM openalex.authors.split_rematched
GROUP BY match_outcome
ORDER BY count DESC

### Step 5: Cluster unmatched and mint new author IDs

Same clustering logic as MatchAuthors Step 3: `xxhash64(normalized_name, institutions|sources)`.

In [0]:
DECLARE OR REPLACE VARIABLE max_id BIGINT;
SET VARIABLE max_id = (SELECT MAX(id) FROM openalex.authors.authors);

CREATE OR REPLACE TABLE openalex.authors.split_new_author_queue AS
WITH unmatched_with_hash AS (
    SELECT
        sr.work_id,
        sr.author_sequence,
        sr.raw_author_name,
        sr.old_author_id,

        xxhash64(
            CASE
                WHEN sr.pn_first IS NOT NULL AND sr.pn_last IS NOT NULL
                THEN CONCAT(sr.pn_first, ' ', sr.pn_last)
                WHEN sr.pn_first_initial IS NOT NULL AND sr.pn_last IS NOT NULL
                THEN CONCAT(sr.pn_first_initial, ' ', sr.pn_last)
                ELSE LOWER(TRIM(sr.raw_author_name))
            END,
            CASE
                WHEN SIZE(sr.all_institution_ids) > 0
                THEN CONCAT_WS('|', SORT_ARRAY(sr.all_institution_ids))
                ELSE CONCAT_WS('|', SORT_ARRAY(sr.work_source_ids))
            END
        ) AS cluster_hash

    FROM openalex.authors.split_rematched sr
    WHERE sr.match_outcome != 'MATCHED'
      AND sr.raw_author_name IS NOT NULL
      AND TRIM(sr.raw_author_name) != ''
),
unique_clusters AS (
    SELECT
        cluster_hash,
        MAX_BY(raw_author_name, LENGTH(raw_author_name)) AS raw_display_name,
        MONOTONICALLY_INCREASING_ID() AS batch_row_id
    FROM unmatched_with_hash
    GROUP BY cluster_hash
)
SELECT
    uc.cluster_hash,
    CASE
        WHEN SIZE(SPLIT(uc.raw_display_name, ',')) = 2 THEN
            TRIM(SPLIT(uc.raw_display_name, ',')[1]) || ' ' || TRIM(SPLIT(uc.raw_display_name, ',')[0])
        ELSE uc.raw_display_name
    END AS display_name,
    max_id + ROW_NUMBER() OVER (ORDER BY uc.batch_row_id) AS new_author_id
FROM unique_clusters uc

### Step 6: Write new author profiles

In [0]:
INSERT INTO openalex.authors.authors
    (id, display_name, full_name, orcid, created_date, updated_date)
SELECT
    new_author_id AS id,
    display_name,
    display_name AS full_name,
    NULL AS orcid,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.authors.split_new_author_queue

### Step 7: Update work_authors with new assignments

For re-matched works, assign the existing_author_id. For newly minted, assign via cluster_hash join.

In [0]:
MERGE INTO openalex.works.work_authors AS target
USING (
    SELECT
        sr.work_id,
        sr.author_sequence,
        COALESCE(
            CASE WHEN sr.match_outcome = 'MATCHED' THEN sr.existing_author_id END,
            nq.new_author_id
        ) AS new_author_id
    FROM openalex.authors.split_rematched sr
    LEFT JOIN openalex.authors.split_new_author_queue nq
        ON sr.match_outcome != 'MATCHED'
        AND xxhash64(
            CASE
                WHEN sr.pn_first IS NOT NULL AND sr.pn_last IS NOT NULL
                THEN CONCAT(sr.pn_first, ' ', sr.pn_last)
                WHEN sr.pn_first_initial IS NOT NULL AND sr.pn_last IS NOT NULL
                THEN CONCAT(sr.pn_first_initial, ' ', sr.pn_last)
                ELSE LOWER(TRIM(sr.raw_author_name))
            END,
            CASE
                WHEN SIZE(sr.all_institution_ids) > 0
                THEN CONCAT_WS('|', SORT_ARRAY(sr.all_institution_ids))
                ELSE CONCAT_WS('|', SORT_ARRAY(sr.work_source_ids))
            END
        ) = nq.cluster_hash
    WHERE COALESCE(
            CASE WHEN sr.match_outcome = 'MATCHED' THEN sr.existing_author_id END,
            nq.new_author_id
          ) IS NOT NULL
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN UPDATE SET
    target.author_id = source.new_author_id,
    target.updated_at = current_timestamp()

### Step 8: Update author full_name for majority clusters

For author profiles where the majority cluster's normalized name differs from the current `full_name`'s normalization, update `full_name` to the longest raw name in the majority cluster.

In [0]:
MERGE INTO openalex.authors.authors AS target
USING (
    SELECT
        sc.author_id,
        MAX_BY(wa.raw_author_name, LENGTH(wa.raw_author_name)) AS best_raw_name
    FROM openalex.authors.split_author_clusters sc
    JOIN openalex.works.work_authors wa ON wa.author_id = sc.author_id
    LEFT JOIN openalex.authors.parsed_names_normalized pn
        ON TRIM(wa.raw_author_name) = pn.raw_author_name
    LEFT JOIN openalex.authors.authors a ON sc.author_id = a.id
    LEFT JOIN openalex.authors.parsed_names_normalized apn
        ON TRIM(a.full_name) = apn.raw_author_name
    WHERE sc.keeps_author_id
      AND CASE
            WHEN pn.normalized_first IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first, ' ', pn.normalized_last)
            WHEN pn.normalized_first_initial IS NOT NULL AND pn.normalized_last IS NOT NULL
            THEN CONCAT(pn.normalized_first_initial, ' ', pn.normalized_last)
            ELSE LOWER(TRIM(wa.raw_author_name))
          END = sc.name_cluster_key
      AND sc.name_cluster_key != CASE
            WHEN apn.normalized_first IS NOT NULL AND apn.normalized_last IS NOT NULL
            THEN CONCAT(apn.normalized_first, ' ', apn.normalized_last)
            WHEN apn.normalized_first_initial IS NOT NULL AND apn.normalized_last IS NOT NULL
            THEN CONCAT(apn.normalized_first_initial, ' ', apn.normalized_last)
            ELSE LOWER(TRIM(a.full_name))
          END
    GROUP BY sc.author_id
) AS source
ON target.id = source.author_id
WHEN MATCHED THEN UPDATE SET
    target.full_name = source.best_raw_name,
    target.display_name = source.best_raw_name,
    target.updated_date = current_timestamp()

### Step 9: Final logging and verification

In [0]:
SELECT 'REMATCHED_TO_EXISTING' AS outcome, COUNT(*) AS count
FROM openalex.authors.split_rematched WHERE match_outcome = 'MATCHED'
UNION ALL
SELECT 'NEW_AUTHORS_MINTED', COUNT(*) FROM openalex.authors.split_new_author_queue
UNION ALL
SELECT 'TOTAL_WORKS_MOVED', COUNT(*) FROM openalex.authors.split_work_removals
UNION ALL
SELECT 'AFFECTED_AUTHORS', COUNT(DISTINCT old_author_id) FROM openalex.authors.split_work_removals
UNION ALL
SELECT 'AUTHORS_FULLNAME_UPDATED', COUNT(*)
FROM openalex.authors.split_author_clusters sc
JOIN openalex.authors.authors a ON sc.author_id = a.id
LEFT JOIN openalex.authors.parsed_names_normalized apn ON TRIM(a.full_name) = apn.raw_author_name
WHERE sc.keeps_author_id
  AND sc.name_cluster_key != CASE
        WHEN apn.normalized_first IS NOT NULL AND apn.normalized_last IS NOT NULL
        THEN CONCAT(apn.normalized_first, ' ', apn.normalized_last)
        WHEN apn.normalized_first_initial IS NOT NULL AND apn.normalized_last IS NOT NULL
        THEN CONCAT(apn.normalized_first_initial, ' ', apn.normalized_last)
        ELSE LOWER(TRIM(a.full_name))
      END

### Step 9b: Spot-check — sample split works

In [0]:
SELECT
    sr.work_id,
    sr.raw_author_name,
    sr.old_author_id,
    sr.match_outcome,
    sr.existing_author_id AS rematched_to,
    nq.new_author_id AS minted_as,
    wa.author_id AS current_author_id
FROM openalex.authors.split_rematched sr
LEFT JOIN openalex.authors.split_new_author_queue nq
    ON sr.match_outcome != 'MATCHED'
    AND xxhash64(
        CASE
            WHEN sr.pn_first IS NOT NULL AND sr.pn_last IS NOT NULL
            THEN CONCAT(sr.pn_first, ' ', sr.pn_last)
            WHEN sr.pn_first_initial IS NOT NULL AND sr.pn_last IS NOT NULL
            THEN CONCAT(sr.pn_first_initial, ' ', sr.pn_last)
            ELSE LOWER(TRIM(sr.raw_author_name))
        END,
        CASE
            WHEN SIZE(sr.all_institution_ids) > 0
            THEN CONCAT_WS('|', SORT_ARRAY(sr.all_institution_ids))
            ELSE CONCAT_WS('|', SORT_ARRAY(sr.work_source_ids))
        END
    ) = nq.cluster_hash
LEFT JOIN openalex.works.work_authors wa
    ON sr.work_id = wa.work_id AND sr.author_sequence = wa.author_sequence
LIMIT 50